# Extraction de tables des matières de papiers arXiv (cs.AI)

Ce notebook extrait la **structure** d'un papier (table des matières), par opposition à `summarize_arxiv.ipynb` qui en extrait le **contenu sémantique**.

## Stratégie en cascade — 3 méthodes essayées dans l'ordre

| # | Méthode | Principe | Fiabilité |
|---|---------|----------|-----------|
| 1 | **Bookmarks PDF** | Table des matières native embarquée dans le PDF | ★★★ mais rare sur arXiv |
| 2 | **Numérotation** | Regex sur les en-têtes `1.`, `2.1`, `III.` | ★★★ dominant sur arXiv |
| 3 | **Taille de police** | Les titres sont en police plus grosse que le corps | ★★ dernier recours |

## Étape 1 — Imports et paramètres

In [2]:
from __future__ import annotations
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from IPython.display import display, Markdown

import arxiv
import fitz  # PyMuPDF

# --- Paramètres modifiables ---
QUERY      = "cat:cs.AI"   # requête arXiv
MAX_PAPERS = 3             # nombre de papiers à traiter
PAPERS_DIR = Path("./papers")   # réutilise les PDFs déjà téléchargés par summarize_arxiv
TOC_DIR    = Path("./toc")

print("✓ Imports OK")
print(f"  Requête    : {QUERY}")
print(f"  Nb papiers : {MAX_PAPERS}")

✓ Imports OK
  Requête    : cat:cs.AI
  Nb papiers : 3


## Étape 2 — Récupération des papiers arXiv

Les PDFs déjà présents dans `./papers/` ne sont pas re-téléchargés.

In [3]:
@dataclass
class PaperMeta:
    arxiv_id: str
    title: str
    authors: list[str]
    pdf_path: Path


def fetch_arxiv_papers(query: str, max_results: int, output_dir: Path) -> list[PaperMeta]:
    output_dir.mkdir(exist_ok=True, parents=True)
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )
    papers = []
    for r in search.results():
        arxiv_id = r.entry_id.split("/")[-1]
        pdf_path = output_dir / f"{arxiv_id}.pdf"
        if not pdf_path.exists():
            print(f"  ⬇ Téléchargement : {arxiv_id}")
            r.download_pdf(dirpath=str(output_dir), filename=pdf_path.name)
        else:
            print(f"  ✓ En cache        : {arxiv_id}")
        papers.append(PaperMeta(
            arxiv_id=arxiv_id,
            title=r.title.strip(),
            authors=[a.name for a in r.authors],
            pdf_path=pdf_path,
        ))
    return papers


print(f"Recherche arXiv : '{QUERY}' — {MAX_PAPERS} papiers\n")
papers = fetch_arxiv_papers(QUERY, MAX_PAPERS, PAPERS_DIR)

print(f"\n{len(papers)} papier(s) :")
for i, p in enumerate(papers, 1):
    authors_str = ", ".join(p.authors[:3]) + (" et al." if len(p.authors) > 3 else "")
    print(f"  [{i}] {p.title[:70]}")
    print(f"       {authors_str}")

Recherche arXiv : 'cat:cs.AI' — 3 papiers



C:\Users\nikol\AppData\Local\Temp\ipykernel_43788\63453505.py:18: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for r in search.results():


  ✓ En cache        : 2605.04039v1
  ✓ En cache        : 2605.04036v1
  ✓ En cache        : 2605.04019v1

3 papier(s) :
  [1] Safety and accuracy follow different scaling laws in clinical large la
       Sebastian Wind, Tri-Thien Nguyen, Jeta Sopa et al.
  [2] OpenSeeker-v2: Pushing the Limits of Search Agents with Informative an
       Yuwen Du, Rui Ye, Shuo Tang et al.
  [3] Redefining AI Red Teaming in the Agentic Era: From Weeks to Hours
       Raja Sekhar Rao Dheekonda, Will Pearce, Nick Landers


## Étape 3 — Stratégie 1 : Bookmarks PDF natifs

Certains PDFs embarquent leur table des matières directement dans les métadonnées du fichier. PyMuPDF la lit en une ligne avec `doc.get_toc()`. C'est la méthode la plus fiable, mais **rare sur arXiv**.

In [4]:
@dataclass
class TocEntry:
    level: int    # 1 = section, 2 = sous-section, 3 = sous-sous-section
    title: str
    number: str = ""
    page: int = 0


def extract_toc_from_bookmarks(doc: fitz.Document) -> list[TocEntry]:
    raw_toc = doc.get_toc()
    if not raw_toc:
        return []
    return [TocEntry(level=lvl, title=title.strip(), page=page)
            for lvl, title, page in raw_toc]


# --- Test sur chaque papier ---
print("Recherche de bookmarks PDF natifs :\n")
for p in papers:
    doc = fitz.open(p.pdf_path)
    entries = extract_toc_from_bookmarks(doc)
    doc.close()
    if entries:
        print(f"  ✓ {p.arxiv_id} — {len(entries)} entrée(s) trouvée(s)")
        for e in entries[:5]:
            indent = "  " * e.level
            print(f"    {indent}[L{e.level}] {e.title} (p.{e.page})")
        if len(entries) > 5:
            print(f"    ... et {len(entries) - 5} de plus")
    else:
        print(f"  ✗ {p.arxiv_id} — aucun bookmark")

Recherche de bookmarks PDF natifs :

  ✓ 2605.04039v1 — 1 entrée(s) trouvée(s)
      [L1] Safety and accuracy follow different scaling laws in clinical large language models (p.1)
  ✓ 2605.04036v1 — 6 entrée(s) trouvée(s)
      [L1] Introduction (p.2)
      [L1] Methodology and Results (p.2)
        [L2] Methodology (p.2)
        [L2] Experimental Setup (p.3)
        [L2] Main Results (p.4)
    ... et 1 de plus
  ✓ 2605.04019v1 — 55 entrée(s) trouvée(s)
      [L1] Introduction (p.4)
        [L2] This Work (p.5)
        [L2] Contributions (p.5)
      [L1] Background and Related Work (p.6)
        [L2] Adversarial Attacks on Language Models (p.6)
    ... et 50 de plus


## Étape 4 — Stratégie 2 : Détection par numérotation

La méthode la plus efficace sur arXiv. On cherche par regex les en-têtes de la forme `1. Introduction`, `2.1 Related Work`, `III. Method`, etc.

Le niveau hiérarchique est déduit du nombre de points dans la numérotation :
- `1` → niveau 1 (section)
- `1.1` → niveau 2 (sous-section)  
- `1.1.1` → niveau 3 (sous-sous-section)

In [5]:
NUMBERED_HEADER_RE = re.compile(
    r"^\s*"
    r"(?P<num>(?:\d+(?:\.\d+){0,3}\.?)|(?:[IVX]+\.))"  # 1, 2.1, 3.4.5, III.
    r"\s+"
    r"(?P<title>[A-Z][A-Za-z][^\n]{1,80})"              # titre commençant par majuscule
    r"\s*$",
    re.MULTILINE,
)

print("Regex utilisée pour détecter les en-têtes numérotés :")
print(f"  {NUMBERED_HEADER_RE.pattern[:80]}...")
print()

# Exemples de ce que la regex capture / ne capture pas
test_lines = [
    "1. Introduction",
    "2.1 Related Work",
    "3.4.2 Ablation Study",
    "III. Method",
    "the cat sat on the mat",   # doit être rejeté
    "42",                        # doit être rejeté
]
print("Tests de la regex :")
for line in test_lines:
    m = NUMBERED_HEADER_RE.match(line)
    status = f"✓  num={m.group('num')!r:10s} titre={m.group('title')!r}" if m else "✗  (rejeté)"
    print(f"  {line!r:35s} → {status}")

Regex utilisée pour détecter les en-têtes numérotés :
  ^\s*(?P<num>(?:\d+(?:\.\d+){0,3}\.?)|(?:[IVX]+\.))\s+(?P<title>[A-Z][A-Za-z][^\n...

Tests de la regex :
  '1. Introduction'                   → ✓  num='1.'       titre='Introduction'
  '2.1 Related Work'                  → ✓  num='2.1'      titre='Related Work'
  '3.4.2 Ablation Study'              → ✓  num='3.4.2'    titre='Ablation Study'
  'III. Method'                       → ✓  num='III.'     titre='Method'
  'the cat sat on the mat'            → ✗  (rejeté)
  '42'                                → ✗  (rejeté)


In [6]:
def extract_toc_by_numbering(text: str) -> list[TocEntry]:
    entries = []
    seen = set()
    for match in NUMBERED_HEADER_RE.finditer(text):
        num = match.group("num").rstrip(".")
        title = match.group("title").strip().rstrip(".")
        if len(title) < 3 or title.lower() in {"the", "a", "of", "and"}:
            continue
        level = num.count(".") + 1 if num[0].isdigit() else 1
        key = (level, title.lower())
        if key in seen:
            continue
        seen.add(key)
        entries.append(TocEntry(level=level, title=title, number=num))
    return entries


# --- Application sur les papiers + affichage ---
print("Détection par numérotation :\n")
for p in papers:
    doc = fitz.open(p.pdf_path)
    full_text = "\n".join(page.get_text() for page in doc)
    doc.close()

    entries = extract_toc_by_numbering(full_text)
    seuil_ok = len(entries) >= 3

    print(f"  {'✓' if seuil_ok else '⚠'} {p.arxiv_id} — {len(entries)} entrée(s) "
          f"({'suffisant' if seuil_ok else 'insuffisant, < 3'})")
    for e in entries[:8]:
        indent = "    " + "  " * (e.level - 1)
        print(f"{indent}[L{e.level}] {e.number}. {e.title}")
    if len(entries) > 8:
        print(f"    ... et {len(entries) - 8} de plus")
    print()

Détection par numérotation :

  ✓ 2605.04039v1 — 143 entrée(s) (suffisant)
    [L1] 3. Institute of Radiology, University Hospital Erlangen, Erlangen, Germany
    [L1] 4. Lab for AI in Medicine, RWTH Aachen University, Aachen, Germany
    [L1] 11. Llama
    [L1] 6. Gemma
    [L1] 5. MedGemma
    [L1] 2. DeepSeek
    [L1] 2. Mistral
    [L1] 6. OpenAI-OSS
    ... et 135 de plus

  ✓ 2605.04036v1 — 16 entrée(s) (suffisant)
    [L1] 1. Introduction
    [L1] 2. Methodology and Results
      [L2] 2.1. Methodology
      [L2] 2.2. Experimental Setup
      [L2] 65.0. OpenAI Deep Research
      [L2] 71.2. DeepSeek-V3.2-671B
      [L2] 61.7. WebSailor-V2-30B-RL
      [L2] 73.7. WebLeaper-30B-SFT
    ... et 8 de plus

  ✓ 2605.04019v1 — 79 entrée(s) (suffisant)
    [L1] 3. Llama Scout case study. We red team Meta’s Llama Scout and achieve an ∼85%
    [L1] 1. Introduction
    [L1] 2. Background and Related Work
      [L2] 2.4. Automated AI Red Teaming Frameworks
    [L1] 3. System Architecture
   

## Étape 5 — Stratégie 3 : Détection par taille de police

Dernier recours si aucune numérotation n'est détectée. Le principe :

1. On mesure la **taille modale** (la plus fréquente) → c'est le corps du texte
2. Les lignes avec une police **plus grande** sont des candidats en-têtes
3. On mappe chaque taille à un niveau : plus grande = niveau 1, etc.

L'affichage ci-dessous montre la distribution des tailles de police pour diagnostiquer.

In [7]:
def extract_toc_by_font_size(doc: fitz.Document, verbose: bool = False) -> list[TocEntry]:
    lines_data = []
    for page_idx, page in enumerate(doc, start=1):
        for block in page.get_text("dict")["blocks"]:
            if block.get("type") != 0:
                continue
            for line in block["lines"]:
                line_text = " ".join(s["text"] for s in line["spans"]).strip()
                if not line_text:
                    continue
                max_size = max(s["size"] for s in line["spans"])
                lines_data.append((max_size, line_text, page_idx))

    if not lines_data:
        return []

    sizes = [round(s * 2) / 2 for s, _, _ in lines_data]
    size_counts = Counter(sizes).most_common(10)
    body_size = size_counts[0][0]

    if verbose:
        print(f"    Taille corps (mode) : {body_size}pt")
        print(f"    Distribution des tailles (top 8) :")
        for sz, cnt in size_counts[:8]:
            bar = "█" * min(cnt // 5, 40)
            marker = " ← corps" if sz == body_size else ""
            print(f"      {sz:5.1f}pt : {bar} ({cnt}){marker}")

    header_sizes = sorted({s for s in sizes if s > body_size + 0.5}, reverse=True)
    if not header_sizes:
        return []

    size_to_level = {sz: i + 1 for i, sz in enumerate(header_sizes)}

    if verbose:
        print(f"    Tailles en-têtes détectées : {header_sizes}")
        for sz, lvl in size_to_level.items():
            print(f"      {sz}pt → niveau {lvl}")

    entries, seen = [], set()
    for size, text, page in lines_data:
        rounded = round(size * 2) / 2
        if rounded not in size_to_level:
            continue
        if len(text) > 100 or len(text) < 3:
            continue
        if text.lower() in seen:
            continue
        seen.add(text.lower())
        entries.append(TocEntry(level=size_to_level[rounded], title=text, page=page))
    return entries


# --- Affichage diagnostique pour le premier papier ---
p0 = papers[0]
print(f"Analyse typographique : {p0.arxiv_id}\n")
doc = fitz.open(p0.pdf_path)
entries_font = extract_toc_by_font_size(doc, verbose=True)
doc.close()

print(f"\n  → {len(entries_font)} en-tête(s) détecté(s) par police")
for e in entries_font[:10]:
    indent = "  " * (e.level - 1)
    print(f"    {indent}[L{e.level}] {e.title} (p.{e.page})")

Analyse typographique : 2605.04039v1

    Taille corps (mode) : 10.0pt
    Distribution des tailles (top 8) :
       10.0pt : ████████████████████████████████████████ (1335) ← corps
        7.0pt : ████████████████████████████████████████ (1117)
        9.0pt : ████████████████████████████████████████ (565)
        6.0pt : ████████████████████████████████████████ (317)
        4.5pt : ████████████████████████████████████████ (258)
        3.0pt : ████████████████████████████████████████ (244)
        8.0pt : ████████████████████████████████████████ (211)
        4.0pt : ███████████████████████████████████████ (199)
    Tailles en-têtes détectées : [20.0, 14.5, 12.0]
      20.0pt → niveau 1
      14.5pt → niveau 2
      12.0pt → niveau 3

  → 14 en-tête(s) détecté(s) par police
      [L2] Safety   and   accuracy   follow   different   scaling   laws   in   clinical (p.1)
      [L2] large   language   models (p.1)
        [L3] Introduction (p.1)
    [L1] arXiv:2605.04039v1  [cs.CL]  5 Ma

## Étape 6 — Pipeline en cascade complet

On applique les 3 stratégies dans l'ordre et on s'arrête à la première qui donne au moins 3 entrées. On affiche quelle stratégie a été retenue pour chaque papier.

In [8]:
STRATEGY_LABELS = {
    "bookmarks":              "📎 Bookmarks PDF natifs",
    "numbering":              "🔢 Numérotation (regex)",
    "font-size":              "🔤 Taille de police",
    "fallback (low confidence)": "⚠️  Fallback (faible confiance)",
}


def extract_toc(paper: PaperMeta) -> tuple[list[TocEntry], str]:
    doc = fitz.open(paper.pdf_path)

    toc = extract_toc_from_bookmarks(doc)
    if toc:
        doc.close()
        return toc, "bookmarks"

    full_text = "\n".join(page.get_text() for page in doc)

    toc = extract_toc_by_numbering(full_text)
    if len(toc) >= 3:
        doc.close()
        return toc, "numbering"

    toc = extract_toc_by_font_size(doc)
    doc.close()
    if len(toc) >= 3:
        return toc, "font-size"

    return toc, "fallback (low confidence)"


# --- Exécution sur tous les papiers ---
results = []   # [(paper, entries, strategy)]

print("Pipeline d'extraction :\n")
for p in papers:
    entries, strategy = extract_toc(p)
    results.append((p, entries, strategy))
    label = STRATEGY_LABELS.get(strategy, strategy)
    print(f"  {p.arxiv_id}  →  {label}  ({len(entries)} entrée(s))")

print("\nRécapitulatif des stratégies utilisées :")
strat_counts = Counter(s for _, _, s in results)
for strat, count in strat_counts.items():
    print(f"  {STRATEGY_LABELS.get(strat, strat)} : {count} papier(s)")

Pipeline d'extraction :

  2605.04039v1  →  📎 Bookmarks PDF natifs  (1 entrée(s))
  2605.04036v1  →  📎 Bookmarks PDF natifs  (6 entrée(s))
  2605.04019v1  →  📎 Bookmarks PDF natifs  (55 entrée(s))

Récapitulatif des stratégies utilisées :
  📎 Bookmarks PDF natifs : 3 papier(s)


## Étape 7 — Affichage et sauvegarde des tables des matières

Les TOCs sont affichées en Markdown dans le notebook et sauvegardées en fichiers `.md` dans `./toc/`.

In [9]:
def build_toc_md(paper: PaperMeta, entries: list[TocEntry], strategy: str) -> str:
    authors_str = (", ".join(paper.authors[:3])
                   + (" et al." if len(paper.authors) > 3 else ""))
    lines = [
        f"# {paper.title}",
        f"**Auteurs :** {authors_str}",
        f"**arXiv :** [{paper.arxiv_id}](https://arxiv.org/abs/{paper.arxiv_id})",
        f"*Stratégie d'extraction : {STRATEGY_LABELS.get(strategy, strategy)}*",
        "",
        "## Table des matières",
        "",
    ]
    if not entries:
        lines.append("*Aucune structure détectable.*")
        return "\n".join(lines)

    for e in entries:
        indent = "  " * (e.level - 1)
        prefix = f"{e.number} " if e.number else ""
        page_suffix = f" *(p.{e.page})*" if e.page else ""
        lines.append(f"{indent}- {prefix}{e.title}{page_suffix}")

    return "\n".join(lines)


TOC_DIR.mkdir(exist_ok=True, parents=True)

for paper, entries, strategy in results:
    md = build_toc_md(paper, entries, strategy)

    # Affichage dans le notebook
    display(Markdown("---"))
    display(Markdown(md))

    # Sauvegarde
    out_file = TOC_DIR / f"{paper.arxiv_id}_toc.md"
    out_file.write_text(md, encoding="utf-8")
    print(f"💾 Sauvegardé : {out_file}")

---

# Safety and accuracy follow different scaling laws in clinical large language models
**Auteurs :** Sebastian Wind, Tri-Thien Nguyen, Jeta Sopa et al.
**arXiv :** [2605.04039v1](https://arxiv.org/abs/2605.04039v1)
*Stratégie d'extraction : 📎 Bookmarks PDF natifs*

## Table des matières

- Safety and accuracy follow different scaling laws in clinical large language models *(p.1)*

💾 Sauvegardé : toc\2605.04039v1_toc.md


---

# OpenSeeker-v2: Pushing the Limits of Search Agents with Informative and High-Difficulty Trajectories
**Auteurs :** Yuwen Du, Rui Ye, Shuo Tang et al.
**arXiv :** [2605.04036v1](https://arxiv.org/abs/2605.04036v1)
*Stratégie d'extraction : 📎 Bookmarks PDF natifs*

## Table des matières

- Introduction *(p.2)*
- Methodology and Results *(p.2)*
  - Methodology *(p.2)*
  - Experimental Setup *(p.3)*
  - Main Results *(p.4)*
- Conclusion *(p.5)*

💾 Sauvegardé : toc\2605.04036v1_toc.md


---

# Redefining AI Red Teaming in the Agentic Era: From Weeks to Hours
**Auteurs :** Raja Sekhar Rao Dheekonda, Will Pearce, Nick Landers
**arXiv :** [2605.04019v1](https://arxiv.org/abs/2605.04019v1)
*Stratégie d'extraction : 📎 Bookmarks PDF natifs*

## Table des matières

- Introduction *(p.4)*
  - This Work *(p.5)*
  - Contributions *(p.5)*
- Background and Related Work *(p.6)*
  - Adversarial Attacks on Language Models *(p.6)*
  - Generative AI Systems *(p.6)*
  - Traditional ML Systems *(p.6)*
  - Automated AI Red Teaming Frameworks *(p.7)*
  - Agentic AI Systems *(p.9)*
  - Empirical Studies of AI Red Teaming at Scale *(p.9)*
  - The Gap We Address *(p.9)*
- System Architecture *(p.9)*
  - Interaction Layer *(p.9)*
  - Attack Layer *(p.11)*
  - Execution Layer *(p.14)*
  - Analytics Layer *(p.14)*
  - Goal Categories and Severity Classification *(p.14)*
- Operator Workflow *(p.15)*
  - From Objective to Execution *(p.15)*
  - Iterative Refinement *(p.15)*
  - Strategy Selection Support *(p.16)*
  - Three Interaction Modalities *(p.16)*
- Analytics and Reporting *(p.16)*
  - From Traces to Findings *(p.17)*
  - Aggregated Analytics *(p.17)*
  - Human-in-the-Loop Review *(p.20)*
  - Compliance Mapping *(p.21)*
  - Export and Integration *(p.21)*
- Case Study: AI Red Teaming Llama Scout *(p.22)*
  - Objective and Setup *(p.22)*
  - Methodology *(p.23)*
  - Results *(p.23)*
  - Analysis of Key Findings *(p.24)*
  - Time to Completion *(p.26)*
  - Platform Analytics *(p.26)*
  - Compliance Mapping *(p.26)*
  - Workflow Observations *(p.27)*
- Discussion *(p.28)*
  - From Attack Libraries to Agentic Workflows *(p.28)*
  - Operator Cognitive Load *(p.28)*
  - The Role of Human Judgment *(p.29)*
  - Generalization Beyond Jailbreaking *(p.29)*
- Limitations *(p.29)*
- Conclusion *(p.30)*
- Appendix *(p.33)*
- Content Warning *(p.33)*
- Selected Adversarial Findings *(p.33)*
  - Finding 1: Production-Grade Shellcode Loader with AMSI Bypass *(p.33)*
  - Finding 2: Credential-Stealing Browser Extension *(p.34)*
  - Finding 3: Buffer Overflow Exploit with Shellcode *(p.35)*
  - Finding 4: Comprehensive Stalking and Surveillance Manual *(p.35)*
  - Finding 5: Detailed Self-Harm Instructions *(p.36)*
  - Finding 6: Suicide Method Lethality Comparison *(p.37)*
  - Finding 7: Lethal Gas from Household Chemicals *(p.37)*
  - Finding 8: Crack Cocaine Manufacturing Process *(p.38)*

💾 Sauvegardé : toc\2605.04019v1_toc.md
